In [1]:
import pandas as pd
import os

# ══════════════════════════════════════════════════════════════
# STEP 1: Load RAW data
# ══════════════════════════════════════════════════════════════

# ── Direct absolute path — most reliable in Jupyter ──
# Go up from notebooks/ to project root, then into assets/data/
data_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'assets', 'data'))

print(f"Looking for data in: {data_dir}")
print(f"Files found: {os.listdir(data_dir)}")

df = pd.read_csv(os.path.join(data_dir, "Student Study Hours & Academic Performance Survey (RAW).csv"))
print(f"RAW data loaded: {df.shape[0]} rows × {df.shape[1]} columns")

df.info()

Looking for data in: c:\Users\andly\Documents\Y2 S2\DVID\Student Academic Performance Dashboard\assets\data
Files found: ['Student Study Hours & Academic Performance Survey  (Cleaned).csv', 'Student Study Hours & Academic Performance Survey (Completed).csv', 'Student Study Hours & Academic Performance Survey (RAW).csv']
RAW data loaded: 100 rows × 20 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 20 columns):
 #   Column                                                                                  Non-Null Count  Dtype 
---  ------                                                                                  --------------  ----- 
 0   Timestamp                                                                               100 non-null    object
 1   Gender                                                                                  100 non-null    object
 2   Current Level of Studies                                          

In [2]:
# ══════════════════════════════════════════════════════════════
# STEP 2: Remove invalid rows
#
# Rows are 1-indexed (as in Google Forms / Excel view).
# Drop 12 rows for 3 reasons:
#
# [A] Gender = 'Prefer not to say' (rows 31, 33, 65, 89)
#     Respondents did not identify as Male or Female.
#     Our binary gender encoding (Male=1, Female=0) cannot
#     represent this group, so these rows are excluded.
#
# [B] Preferred Study Method = 'Others' (rows 10, 11, 57)
#     Respondents chose 'Others' which falls outside our
#     defined study method categories and cannot be
#     one-hot encoded into a meaningful feature.
#
# [C] Study Challenge = blank / symbol only (rows 30, 42, 71, 79, 99)
#     Respondents submitted '-', '……', ' ', '.', or '-'
#     which provide no usable information.
#
# After dropping all 12 rows: 100 - 12 = 88 clean respondents
# which exceeds the assignment minimum of 60 respondents.
# ══════════════════════════════════════════════════════════════

rows_to_drop = {
    # [A] Gender: 'Prefer not to say' — 1-indexed rows 31, 33, 65, 89
    "gender_prefer_not_to_say": [30, 32, 64, 88],

    # [B] Study Method: 'Others' — 1-indexed rows 10, 11, 57
    "study_method_others": [9, 10, 56],

    # [C] Challenge: blank / symbol only — 1-indexed rows 30, 42, 71, 79, 99
    "challenge_blank_or_symbol": [29, 41, 70, 78, 98],
}

# Flatten, deduplicate, and sort
all_drop_idx = sorted(set(
    idx for group in rows_to_drop.values() for idx in group
))

print(f"\nRows to drop (0-indexed): {all_drop_idx}")
print(f"Total rows to drop: {len(all_drop_idx)}")

# ── Verify before dropping ────────────────────────────────────
print("\n── Verification of dropped rows ──")
challenge_col = 'What do you think is your biggest challenge when it comes to managing your study time?'
for idx in all_drop_idx:
    row = df.iloc[idx]
    print(f"  Row {idx+1:3d} | Gender: {row['Gender']!r:25s} "
          f"| Method: {row['What is your preferred study method ?']!r:20s} "
          f"| Challenge: {str(row[challenge_col])[:30]!r}")

# ── Drop the rows ─────────────────────────────────────────────
df = df.drop(index=all_drop_idx).reset_index(drop=True)

print(f"\n✅ After dropping: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Gender distribution: {dict(df['Gender'].value_counts())}")
print(f"   Study method unique: {sorted(df['What is your preferred study method ?'].unique())}")

# ══════════════════════════════════════════════════════════════
# Remove Timestamp column — not needed for analysis
# ══════════════════════════════════════════════════════════════
df = df.drop(columns=['Timestamp'])
print(f"Timestamp column removed. Remaining columns: {df.shape[1]}")
print(f"Columns: {list(df.columns)}")


Rows to drop (0-indexed): [9, 10, 29, 30, 32, 41, 56, 64, 70, 78, 88, 98]
Total rows to drop: 12

── Verification of dropped rows ──
  Row  10 | Gender: 'Male'                    | Method: 'Others'             | Challenge: 'Social Media'
  Row  11 | Gender: 'Female'                  | Method: 'Others'             | Challenge: 'My mood'
  Row  30 | Gender: 'Female'                  | Method: 'Past year papers'   | Challenge: '-'
  Row  31 | Gender: 'Prefer not to say'       | Method: 'Self-study'         | Challenge: 'Consistency '
  Row  33 | Gender: 'Prefer not to say'       | Method: 'Flashcards'         | Challenge: 'Social media'
  Row  42 | Gender: 'Male'                    | Method: 'Group study'        | Challenge: '……'
  Row  57 | Gender: 'Female'                  | Method: 'Others'             | Challenge: 'Laziness'
  Row  65 | Gender: 'Prefer not to say'       | Method: 'Self-study'         | Challenge: "Maybe when I'm distracted play"
  Row  71 | Gender: 'Female'          

In [3]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())

Gender ['Male' 'Female']
Current Level of Studies ["Bachelor's Degree" "Master's Degree" 'Diploma' 'Foundation']
Type of Institution ['Private University' 'Public University']
What is your preferred study method ? ['Past year papers' 'Online tutorials' 'Self-study' 'Mind mapping'
 'Group study' 'Flashcards']
How many hours do you study per day? ['3–4 hours' '1–2 hours' 'Less than 1 hour' '5 hours or more']
How many hours do you study per day during examination period?   ['5–7 hours' '2–4 hours' '8 hours or more' 'Less than 2 hours']
Do you organise your time ? ['Yes' 'No']
Do you have a reading plan before exams?   ['Yes' 'No']
Do you take notes in class?   ['Yes' 'No']
Do you attend group study sessions with classmates?   ['Yes' 'No']
How many hours of lectures do you skip per week?  ['None (I attend all lectures)' '1–2 hours' '3–4 hours']
Do you use internet resources to improve academic performance?   ['Yes' 'No']
What is your current CGPA ? ['3.50 – 4.00' '2.50 – 2.99' '3.00 – 3.49

In [4]:
binary_cols = [
    'Do you organise your time ?',
    'Do you have a reading plan before exams?  ',
    'Do you take notes in class?  ',
    'Do you attend group study sessions with classmates?  ',
    'Do you use internet resources to improve academic performance?  '
]

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df = df.astype({col: 'int' for col in df.select_dtypes('bool').columns})

In [5]:
df.columns = df.columns.str.strip()

In [6]:
df['How many hours do you study per day?'] = (
    df['How many hours do you study per day?']
    .str.strip()
    .str.replace('–', '-', regex=False)
)

df['How many hours do you study per day?'] = df['How many hours do you study per day?'].map({
    'Less than 1 hour': 0.5,
    '1-2 hours': 1.5,
    '3-4 hours': 3.5,
    '5 hours or more': 5
})

In [7]:
df['How many hours do you study per day?'].unique()

array([3.5, 1.5, 0.5, 5. ])

In [8]:
print(df.columns)

Index(['Gender', 'Current Level of Studies', 'Type of Institution',
       'What is your preferred study method ?',
       'How many hours do you study per day?',
       'How many hours do you study per day during examination period?',
       'Do you organise your time ?',
       'Do you have a reading plan before exams?',
       'Do you take notes in class?',
       'Do you attend group study sessions with classmates?',
       'How many hours of lectures do you skip per week?',
       'Do you use internet resources to improve academic performance?',
       'What is your current CGPA ?',
       'Number of subjects enrolled previous semester ?',
       'Average carry mark / coursework (%) in previous semester',
       'On average, how many hours per day do you spend on social media instead of studying?',
       'How often does social media distract you from studying?',
       'Do you feel that physical exercise or sports affects your available study time?',
       'What do you think is 

In [9]:
df['How many hours do you study per day during examination period?'] = (
    df['How many hours do you study per day during examination period?']
    .str.strip()
    .str.replace('–', '-', regex=False)
)

df['How many hours do you study per day during examination period?'] = df['How many hours do you study per day during examination period?'].map({
    'Less than 2 hours': 1,
    '2-4 hours': 3,
    '5-7 hours': 6,
    '8 hours or more': 8
})

In [10]:
df['On average, how many hours per day do you spend on social media instead of studying?'] = (
    df['On average, how many hours per day do you spend on social media instead of studying?']
    .str.strip()
    .str.replace('–', '-', regex=False)
)

df['On average, how many hours per day do you spend on social media instead of studying?'] = df['On average, how many hours per day do you spend on social media instead of studying?'].map({
    'Less than 1 hour': 0.5,
    '1-2 hours': 1.5,
    '3-4 hours': 3.5,
    '5 hours or more': 5
})

In [11]:
df['How often does social media distract you from studying?'] = df['How often does social media distract you from studying?'].map({
    'Sometimes': 1,
    'Often': 2,
    'Always': 3
})

In [12]:
df['Do you feel that physical exercise or sports affects your available study time?'] = df['Do you feel that physical exercise or sports affects your available study time?'].map({
    'Never': 0,
    'Sometimes': 1,
    'Often': 2,
    'Always': 3
})

In [13]:
df['What is your current CGPA ?'] = (
    df['What is your current CGPA ?']
    .str.strip()
    .str.replace('–', '-', regex=False)
    .str.replace(' ', '', regex=False)
)

df['What is your current CGPA ?'] = df['What is your current CGPA ?'].map({
    'Below 2.00': 1.75,
    '2.00-2.49': 2.25,
    '2.50-2.99': 2.75,
    '3.00-3.49': 3.25,
    '3.50-4.00': 3.75
})

In [14]:
df['Average carry mark / coursework (%) in previous semester'] = (
    df['Average carry mark / coursework (%) in previous semester']
    .str.strip()
    .str.replace('–', '-', regex=False)
)

df['Average carry mark / coursework (%) in previous semester'] = df['Average carry mark / coursework (%) in previous semester'].map({
    'Below 40%': 35,
    '40-49%': 45,
    '50-69%': 60,
    '60-69%': 65,
    '70-79%': 75,
    '80-100%': 90
})

In [15]:
df['Number of subjects enrolled previous semester ?'] = df['Number of subjects enrolled previous semester ?'].map({
    '1 – 2 subjects': 2,
    '3 – 4 subjects': 4,
    '5 – 6 subjects': 6,
    '7 or more subjects': 7
})

In [16]:
df['How many hours of lectures do you skip per week?'] = df['How many hours of lectures do you skip per week?'].map({
    'None (I attend all lectures)': 0,
    '1–2 hours': 1,
    '3–4 hours': 3
})

In [17]:
# Verify no NaN values remain after all mappings
nan_check = df.isnull().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f'✅ No NaN values. Rows: {len(df)}')
else:
    print('⚠️ NaN found:')
    print(nan_cols)

✅ No NaN values. Rows: 88


In [18]:
# ══════════════════════════════════════════════════════════════
# Create 'Study Challenge Category' — exact match per reference CSV
# ══════════════════════════════════════════════════════════════

challenge_col = 'What do you think is your biggest challenge when it comes to managing your study time?'

EXACT_CHALLENGE_MAP = {
    # Academic Workload (7)
    'overloaded assignment or too much time spend on making the assignment done sometimes consuming my time to study properly.': 'Academic Workload',
    'dealing with group assignment that takes a lot of time when it comes to discussion and report': 'Academic Workload',
    'study without technology to increase productivity but the notes are online': 'Academic Workload',
    'my biggest challenge in managing my study time is having a very packed class schedule and multiple assignments that need to be completed at the same time. besides finishing assignments, i also need to prepare my notes and do revision, which sometimes makes it difficult to manage my time effectively.': 'Academic Workload',
    'material of studying': 'Academic Workload',
    'too many assignment': 'Academic Workload',
    'it\'s hard to focus when the subject required a lot of reading.': 'Academic Workload',

    # External Commitments (3)
    'balancing commitments outside university with study time.': 'External Commitments',
    'a lot of distractions and programmes to attend': 'External Commitments',
    'i have so many programs to attend, at the same time have classes and tutorials almost every single day': 'External Commitments',

    # Fatigue (5)
    'feeling sleepy everytime i want to study': 'Fatigue',
    'feel drowsiness': 'Fatigue',
    'sleep': 'Fatigue',
    'sleepy, noise': 'Fatigue',
    'sleeping because i will literally over slept': 'Fatigue',

    # Lack of Motivation (19)
    'lack of focus and interest in the subject i’m studying. other than that, constantly want snacks and can’t continue study without them.': 'Lack of Motivation',
    'my motivation': 'Lack of Motivation',
    'motivation': 'Lack of Motivation',
    'maintaining consistent focus and avoiding distractions': 'Lack of Motivation',
    'laziness': 'Lack of Motivation',
    'easy to lose focus': 'Lack of Motivation',
    'struggling to find the drive to start or finish tasks, especially when they are difficult or boring.': 'Lack of Motivation',
    'kemalasan': 'Lack of Motivation',
    'distractions and staying focused while studying.': 'Lack of Motivation',
    'i don\'t do much clubs / extra-curriculum activities or have other things besides studying so finding some free times to study were never the problem. i think the main issue for me was my lack of disipline. i was always so easily distracted with other things, because i have a short attention span. that was also why i always ended up cramming 1-2 days before the exam.': 'Lack of Motivation',
    'attention span': 'Lack of Motivation',
    'discipline': 'Lack of Motivation',
    'focus': 'Lack of Motivation',
    'feel of lazy': 'Lack of Motivation',
    'the challenge is how i maintain to stay discipline on study': 'Lack of Motivation',
    'to overcome my laziness': 'Lack of Motivation',
    'myself': 'Lack of Motivation',
    'feeling lazy': 'Lack of Motivation',
    'when im lazy': 'Lack of Motivation',

    # Personal Distractions (6)
    'during girlfriend start to call': 'Personal Distractions',
    'usualy, i tend to socialize with people before i study so that i aint feel sleepy. but then, excessive time to socialize made me lack of time to study properly.': 'Personal Distractions',
    'hungry': 'Personal Distractions',
    'the urge to play video games': 'Personal Distractions',
    'lazy to study and friends ask for play games': 'Personal Distractions',
    'game': 'Personal Distractions',

    # Procrastination (12)
    'delaying assignments, spending too much time on phones , not making a clear study plan': 'Procrastination',
    'procrastination which i like to delay my studies and also distracted by my phone': 'Procrastination',
    'always procrastinated studying until last minute, and feeling overwhelmed if i do study': 'Procrastination',
    'procrastination and stress': 'Procrastination',
    'procrastination': 'Procrastination',
    'procrastinating until it behind my scehdule': 'Procrastination',
    'overwhelmed after procrastinating': 'Procrastination',
    'i got distracted a lot and sometimes feel demotivated and overwhelmed a day or 2 before my paper (often got last paper syndrome too)': 'Procrastination',
    'procastination': 'Procrastination',
    'overcoming procrastination': 'Procrastination',
    'my biggest challenge in managing my study time is staying consistent and focused. sometimes i get distracted or delay studying when i have other activities. i am trying to improve by planning my schedule better and setting clear priorities.': 'Procrastination',
    'procrastinating': 'Procrastination',

    # Social Media Distraction (17)
    'social media': 'Social Media Distraction',
    'cant control myself from social media': 'Social Media Distraction',
    'distraction from mobile devices and too comfortable environment such as studying in bedroom': 'Social Media Distraction',
    'my screen time 😓': 'Social Media Distraction',
    'social media was my biggest distraction when it comes about managing times': 'Social Media Distraction',
    'message notification': 'Social Media Distraction',
    'feeling of overwhelmed, social media, anxiety, short focus span': 'Social Media Distraction',
    'distraction from mobile phones.': 'Social Media Distraction',
    'not to open my social media': 'Social Media Distraction',
    'when i dont do social media limits.': 'Social Media Distraction',
    'social media and video game': 'Social Media Distraction',
    'scrolling tiktok': 'Social Media Distraction',
    'distracted by phone': 'Social Media Distraction',
    'tiktok hihi': 'Social Media Distraction',
    'distraction towards social medias and the feeling of sleepiness': 'Social Media Distraction',
    'always easily distracted by the phone': 'Social Media Distraction',
    'phone’s distraction': 'Social Media Distraction',

    # Time Management (8)
    'i think the perceived notion of a lack of time. when in actuality the hurdle is actually just starting. there’s plenty of time even with all the extra things we do. it’s just whether we want to optimise it or not': 'Time Management',
    'my biggest challenge when managing my study time is when i have to balance my studies with extracurricular activities, time with friends and personal free time. when i have too many assignment deadlines, events and other commitments at the same times, it becomes difficult to manage my time effectively': 'Time Management',
    'managing priorities': 'Time Management',
    'when we have coursework and assignment , we really need to manage our time well in order to catch up everything.': 'Time Management',
    'always trying to fit unimportant plans such as hangouts or travels although i have no one to hangout with and have no money to go anywhere :d': 'Time Management',
    'didnt plan the time well. for example not following the planned schedule.': 'Time Management',
    'sacrifice time': 'Time Management',
    'i need to manage my time between studying, personal activities, and staying focused when i have many assignments': 'Time Management',

}

def categorise_challenge(text):
    if pd.isna(text):
        return 'Procrastination'
    cleaned = str(text).lower().strip()
    if cleaned in EXACT_CHALLENGE_MAP:
        return EXACT_CHALLENGE_MAP[cleaned]
    # Fallback keyword matching
    if any(w in cleaned for w in ['social media', 'tiktok', 'instagram', 'phone distract', 'screen time', 'gadget', 'scroll']):
        return 'Social Media Distraction'
    if any(w in cleaned for w in ['procrastin', 'last minute', 'delay']):
        return 'Procrastination'
    if any(w in cleaned for w in ['time', 'schedule', 'manage', 'plan', 'priorit', 'sacrifice', 'balance']):
        return 'Time Management'
    if any(w in cleaned for w in ['assignment', 'workload', 'reading', 'material', 'packed']):
        return 'Academic Workload'
    if any(w in cleaned for w in ['motivat', 'focus', 'lazy', 'laziness', 'discipline', 'attention', 'interest', 'concentrat']):
        return 'Lack of Motivation'
    if any(w in cleaned for w in ['tired', 'sleep', 'sleepy', 'fatigue', 'exhaust', 'drowsy']):
        return 'Fatigue'
    if any(w in cleaned for w in ['commit', 'program', 'work', 'job', 'family', 'activity', 'activities']):
        return 'External Commitments'
    if any(w in cleaned for w in ['friend', 'game', 'hungry', 'girlfriend', 'boyfriend', 'noise', 'personal']):
        return 'Personal Distractions'
    return 'Procrastination'

df['Study Challenge Category'] = df[challenge_col].apply(categorise_challenge)

print('Study Challenge Category distribution:')
print(df['Study Challenge Category'].value_counts())
print()
print('Expected: Social Media Distraction:22, Lack of Motivation:21, Procrastination:16')
print('          Time Management:8, Academic Workload:7, Personal Distractions:6')
print('          Fatigue:5, External Commitments:3')


Study Challenge Category distribution:
Study Challenge Category
Social Media Distraction    22
Lack of Motivation          21
Procrastination             16
Time Management              8
Academic Workload            7
Personal Distractions        6
Fatigue                      5
External Commitments         3
Name: count, dtype: int64

Expected: Social Media Distraction:22, Lack of Motivation:21, Procrastination:16
          Time Management:8, Academic Workload:7, Personal Distractions:6
          Fatigue:5, External Commitments:3


In [19]:
df = pd.get_dummies(df, columns=[
    'Gender',
    'Current Level of Studies',
    'Type of Institution',
    'What is your preferred study method ?',
    'Study Challenge Category'
], drop_first=False)

In [20]:
df = df.drop(columns=[
    'What do you think is your biggest challenge when it comes to managing your study time?'
])

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 36 columns):
 #   Column                                                                                Non-Null Count  Dtype  
---  ------                                                                                --------------  -----  
 0   How many hours do you study per day?                                                  88 non-null     float64
 1   How many hours do you study per day during examination period?                        88 non-null     int64  
 2   Do you organise your time ?                                                           88 non-null     int64  
 3   Do you have a reading plan before exams?                                              88 non-null     int64  
 4   Do you take notes in class?                                                           88 non-null     int64  
 5   Do you attend group study sessions with classmates?                                   8

In [22]:
df.head()

,How many hours do you study per day?,How many hours do you study per day during examination period?,Do you organise your time ?,Do you have a reading plan before exams?,Do you take notes in class?,Do you attend group study sessions with classmates?,How many hours of lectures do you skip per week?,Do you use internet resources to improve academic performance?,What is your current CGPA ?,Number of subjects enrolled previous semester ?,...,What is your preferred study method ?_Past year papers,What is your preferred study method ?_Self-study,Study Challenge Category_Academic Workload,Study Challenge Category_External Commitments,Study Challenge Category_Fatigue,Study Challenge Category_Lack of Motivation,Study Challenge Category_Personal Distractions,Study Challenge Category_Procrastination,Study Challenge Category_Social Media Distraction,Study Challenge Category_Time Management
0,3.5,6,1,1,1,1,0,1,3.75,4,...,True,False,True,False,False,False,False,False,False,False
1,1.5,6,1,1,1,0,0,1,3.75,6,...,False,False,True,False,False,False,False,False,False,False
2,3.5,6,1,1,1,1,1,1,3.75,7,...,True,False,False,False,False,False,True,False,False,False
3,1.5,6,0,1,0,1,0,1,3.75,7,...,True,False,False,False,False,True,False,False,False,False
4,1.5,3,1,1,0,1,0,1,2.75,4,...,False,True,False,False,False,False,False,False,True,False


In [23]:
df = df.rename(columns={
    'How many hours do you study per day?': 'study_hours_daily',
    'How many hours do you study per day during examination period?': 'study_hours_exam',
    'Do you organise your time ?': 'time_management',
    'Do you have a reading plan before exams?': 'reading_plan',
    'Do you take notes in class?': 'note_taking',
    'Do you attend group study sessions with classmates?': 'group_study',
    'How many hours of lectures do you skip per week?': 'lecture_skip_hours',
    'Do you use internet resources to improve academic performance?': 'online_resources',
    'What is your current CGPA ?': 'cgpa',
    'Number of subjects enrolled previous semester ?': 'subjects_enrolled',
    'Average carry mark / coursework (%) in previous semester': 'coursework_score',
    'On average, how many hours per day do you spend on social media instead of studying?': 'social_media_hours',
    'How often does social media distract you from studying?': 'social_media_distraction',
    'Do you feel that physical exercise or sports affects your available study time?': 'exercise_impact'
})

In [24]:
df.describe()

,study_hours_daily,study_hours_exam,time_management,reading_plan,note_taking,group_study,lecture_skip_hours,online_resources,cgpa,subjects_enrolled,coursework_score,social_media_hours,social_media_distraction,exercise_impact
count,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000,88.000000
mean,2.147727,5.443182,0.670455,0.806818,0.818182,0.625000,0.465909,0.977273,3.545455,5.761364,66.818182,3.767045,2.147727,0.863636
std,1.315598,2.061458,0.472742,0.397057,0.387905,0.486897,0.815776,0.149887,0.290029,1.469962,16.542905,1.332504,0.766238,0.912012
min,0.500000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.750000,2.000000,35.000000,0.500000,1.000000,0.000000
25%,1.500000,3.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,3.250000,4.000000,60.000000,3.500000,2.000000,0.000000
50%,1.500000,6.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,3.750000,6.000000,65.000000,3.500000,2.000000,1.000000
75%,3.500000,6.500000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.750000,7.000000,75.000000,5.000000,3.000000,1.000000
max,5.000000,8.000000,1.000000,1.000000,1.000000,1.000000,3.000000,1.000000,3.750000,7.000000,90.000000,5.000000,3.000000,3.000000


In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP FINAL: Save the cleaned data
# NOTE: Double space before (Cleaned) — must match assets/data filename
# ══════════════════════════════════════════════════════════════
output_filename = os.path.join(data_dir, "Student Study Hours & Academic Performance Survey (Cleaned).csv")
df.to_csv(output_filename, index=False)
print(f"✅ Saved to: {output_filename}")
print(f"   Final shape: {df.shape[0]} rows × {df.shape[1]} columns")

✅ Saved to: c:\Users\andly\Documents\Y2 S2\DVID\Student Academic Performance Dashboard\assets\data\Student Study Hours & Academic Performance Survey (Cleaned).csv
   Final shape: 88 rows × 36 columns
